In [0]:
%pip install -qqqq -U mlflow-skinny[databricks] databricks-sdk
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
default_warehouse = next(
    (
        wh
        for wh in w.warehouses.list()
        if "Serverless Starter Warehouse" in wh.name and wh.enable_serverless_compute
    ),
    None,
)
default_warehouse_id = default_warehouse.id if default_warehouse else None
print(f"{default_warehouse_id=}")

In [0]:
%sql
-- drop schema if exists workshop_guy_catalog.mlflow_traces_test cascade

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workshop_guy_catalog.genie_traces_bo

In [0]:
# Example values for the placeholders below:
# MLFLOW_TRACING_SQL_WAREHOUSE_ID: "abc123def456" (found in SQL warehouse URL)
# experiment_name: "/Users/user@company.com/traces"
# catalog_name: "main" or "my_catalog"
# schema_name: "mlflow_traces" or "production_traces"
# table_prefix: "my_otel"

import os
import mlflow
from mlflow.entities.trace_location import UnityCatalog

mlflow.set_tracking_uri("databricks")

# Specify the ID of a SQL warehouse you have access to.
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = default_warehouse_id
# Specify the name of the MLflow Experiment to use for viewing traces in the UI.
experiment_name = "/Workspace/Shared/genie-uc-eval-traces"
# Specify the name of the Catalog to use for storing traces.
catalog_name = "workshop_guy_catalog"
# Specify the name of the Schema to use for storing traces.
schema_name = "genie_traces_bo"
# Specify the name of the prefix appended to every table storing trace data.
table_prefix = "evals"

# mlflow.set_experiment is an upsert operation
experiment = mlflow.set_experiment(
    experiment_name=experiment_name,
    trace_location=UnityCatalog(
        catalog_name=catalog_name,
        schema_name=schema_name,
        table_prefix=table_prefix,  # defaults to experiment id if not provided
    ),
)

print(f"Experiment ID: {experiment.experiment_id}")
print(experiment.trace_location.full_otel_spans_table_name)

In [0]:
experiment.experiment_id